# Task 5: Source Metadata Ingestion into MongoDB

**Mục tiêu**: Xây dựng **Spark Structured Streaming** job tiêu thụ Kafka topic `code.events.metadata` và nạp vào MongoDB collection `source_metadata`, đảm bảo **idempotent** (ghi đè theo `file_path`, không trùng lặp) và **fault-tolerant** (checkpoint offset).

**Thành phần triển khai**:

| File | Vai trò |
|---|---|
| `spark-mongo/metadata_to_mongodb.py` | Spark job chạy trong Docker container |
| `docker-compose.override.yml` | Thêm services: MongoDB, Mongo Express, Spark |
| Notebook này | **Xác minh kết quả** bằng `pymongo` |


## 1. Kiến trúc & Phương pháp tiếp cận

### Luồng xử lý
1. **Kafka Consumer (Spark Streaming)**: Kết nối với Kafka topic `code.events.metadata` ở chế độ streaming (`spark.readStream.format("kafka")`).
2. **Schema Enforcement & Validation**: Ép kiểu dữ liệu chuỗi JSON trong Kafka value sang `METADATA_SCHEMA` (StructType). Lọc và kiểm tra tính toàn vẹn của dữ liệu (chuẩn SHA-256 `file_hash`, `schema_version == 'v1'`, timestamp hợp lệ, số dòng/node/edge không âm).
3. **Micro-batch Deduplication**: Trong mỗi micro-batch, sử dụng window function `row_number()` theo `file_path` để chọn ra bản ghi mới nhất (dựa trên `event_timestamp` và `kafka_offset`).
4. **Idempotent Upsert vào MongoDB**: Sử dụng phương thức `writeStream.foreachBatch()` kết hợp cấu hình `operationType = replace`, `idFieldList = file_path`, và `upsertDocument = true` của MongoDB Spark Connector (v10.3.0) để cập nhật (replace/upsert) tài liệu theo khóa `file_path`.
5. **Checkpointing**: Đặt vị trí lưu `checkpointLocation` lưu giữ thông tin offset đã xử lý, giúp Spark khôi phục chính xác trạng thái từ offset cuối cùng khi restart.

## 2. Các quyết định thiết kế

### 2.1 Schema Validation

Trước khi ghi MongoDB, Spark lọc nghiêm ngặt theo hợp đồng dữ liệu `metadata_event.schema.json`:

| Rule | Mục đích |
|---|---|
| `schema_version == "v1"` | Forward compatibility — bỏ qua schema tương lai |
| `file_hash` khớp regex `^[0-9a-f]{64}$` | Đảm bảo SHA-256 hợp lệ (quan trọng cho Task 6 replay) |
| `kafka_key == file_path` | Phát hiện message bị corrupt key |
| `loc, num_nodes, num_edges ≥ 0` | Không cho giá trị âm vô nghĩa |
| `event_timestamp` parse được | Loại timestamp lỗi format |

### 2.2 Idempotent Upsert

MongoDB Spark Connector (v10.3.0) được cấu hình:

```
operationType  = replace       → ghi đè toàn bộ document
idFieldList    = file_path     → match document theo file_path
upsertDocument = true          → chưa có → insert, có rồi → replace
```

→ Parse lại cùng file bao nhiêu lần cũng chỉ có **1 document** trong MongoDB.

### 2.3 Micro-batch Deduplication

Trong cùng một micro-batch, có thể có nhiều event cho cùng `file_path` (ví dụ parser chạy 2 lần nhanh). Dùng `row_number()` partition by `file_path`, order by `event_timestamp DESC, kafka_offset DESC` để chỉ giữ **bản mới nhất**.

### 2.4 Checkpoint

- Path: `/opt/spark-checkpoints/metadata-to-mongodb` (mount ra host `./spark-checkpoints/`)
- Spark lưu offset Kafka đã xử lý → restart chỉ đọc message **mới**, không reprocess
- Kết hợp với upsert → **double protection** chống trùng lặp

## 3. Triển khai & Chạy

Toàn bộ pipeline chạy qua Docker Compose (file `docker-compose.override.yml`):

| Service | Image | Port |
|---|---|---|
| `mongodb` | `mongo:7.0` | 27017 |
| `mongo-express` | `mongo-express:1.0.2-20` | 8081 |
| `spark-metadata-to-mongodb` | `apache/spark:3.5.1` | — |

```bash
# 1. Dựng services
docker compose up -d

# 2. Seed dữ liệu (parser chạy trên host → Kafka → Spark tự consume)
python parser-service/parser.py --limit 30 --publish

# 3. Xem Spark log
docker compose logs -f spark-metadata-to-mongodb
# → Thấy "MongoDB upsert completed for micro-batch N" = thành công
```

## 4. Xác minh dữ liệu trong MongoDB

Các cell dưới đây kết nối trực tiếp tới MongoDB (qua `pymongo`) để xác minh pipeline hoạt động đúng end-to-end.

### 4.1 Kết nối & Lấy dữ liệu mẫu

In [17]:
from pymongo import MongoClient
import json

MONGODB_URI = "mongodb://localhost:27017"
DATABASE = "cpg"
COLLECTION = "source_metadata"

client = MongoClient(MONGODB_URI)
db = client[DATABASE]
collection = db[COLLECTION]

total_docs = collection.count_documents({})
print(f"Kết nối MongoDB thành công!")
print(f"Tổng số documents trong '{COLLECTION}': {total_docs}")

print(f"\n--- MẪU DỮ LIỆU (3 documents đầu) ---")
for i, doc in enumerate(collection.find().limit(3), 1):
    doc["_id"] = str(doc["_id"])
    print(f"\nDocument {i}:")
    print(json.dumps(doc, indent=2, ensure_ascii=False))


Kết nối MongoDB thành công!
Tổng số documents trong 'source_metadata': 1543

--- MẪU DỮ LIỆU (3 documents đầu) ---

Document 1:
{
  "_id": "6a601f031eba02b8cc5b486c",
  "schema_version": "v1",
  "event_timestamp": "2026-07-22T01:41:00Z",
  "file_path": "demo/task5_test.py",
  "file_hash": "bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb",
  "language": "python",
  "loc": 15,
  "num_nodes": 24,
  "num_edges": 31,
  "repo": "task5-test",
  "repo_commit": "updatedcommit"
}

Document 2:
{
  "_id": "6a60331ac40a10e1f621ef6b",
  "schema_version": "v1",
  "event_timestamp": "2026-07-22T02:55:00Z",
  "file_path": "demo/task5_valid.py",
  "file_hash": "cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc",
  "language": "python",
  "loc": 9,
  "num_nodes": 14,
  "num_edges": 18,
  "repo": "task5-test",
  "repo_commit": "validcommit"
}

Document 3:
{
  "_id": "6a61cfc60659c8b0a3e5e487",
  "schema_version": "v1",
  "event_timestamp": "2026-07-23T08:41:02.199378+00:00"

### 4.2 Kiểm tra tính duy nhất (Idempotency)

Mỗi `file_path` phải có đúng **1 document**. Nếu pipeline replay, bản cũ phải bị ghi đè.

In [18]:
pipeline = [
    {"$group": {"_id": "$file_path", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 1}}}
]
duplicates = list(collection.aggregate(pipeline))

if not duplicates:
    print(f"PASSED: Không có bản ghi trùng lặp")
    print(f"   {total_docs} documents, {total_docs} file_path duy nhất")
else:
    print(f"FAILED: {len(duplicates)} file_path bị trùng:")
    for d in duplicates[:10]:
        print(f"  - {d['_id']}: {d['count']} bản ghi")


PASSED: Không có bản ghi trùng lặp
   1543 documents, 1543 file_path duy nhất


### 4.3 Kiểm tra chất lượng dữ liệu (Schema v1)

Xác minh tất cả documents tuân theo `metadata_event.schema.json`.

In [19]:
import re

SHA256_RE = re.compile(r"^[0-9a-f]{64}$")

invalid_docs = []
for doc in collection.find():
    issues = []
    if doc.get("schema_version") != "v1":
        issues.append(f"schema_version={doc.get('schema_version')!r}")
    if not SHA256_RE.match(doc.get("file_hash", "")):
        issues.append("file_hash invalid")
    if doc.get("language") != "python":
        issues.append(f"language={doc.get('language')!r}")
    for field in ("loc", "num_nodes", "num_edges"):
        val = doc.get(field)
        if val is None or val < 0:
            issues.append(f"{field}={val!r}")
    if not doc.get("file_path"):
        issues.append("file_path missing")
    if not doc.get("repo_commit"):
        issues.append("repo_commit missing")
    if issues:
        invalid_docs.append({"file_path": doc.get("file_path", "?"), "issues": issues})

if not invalid_docs:
    print(f"PASSED: Tất cả {total_docs} documents hợp lệ theo schema v1")
else:
    print(f"FAILED: {len(invalid_docs)}/{total_docs} documents không hợp lệ:")
    for d in invalid_docs[:5]:
        print(f"  - {d['file_path']}: {', '.join(d['issues'])}")


PASSED: Tất cả 1543 documents hợp lệ theo schema v1


### 4.4 Thống kê tổng quan

In [20]:
stats_pipeline = [
    {"$group": {
        "_id": None,
        "total_files": {"$sum": 1},
        "total_loc": {"$sum": "$loc"},
        "total_nodes": {"$sum": "$num_nodes"},
        "total_edges": {"$sum": "$num_edges"},
        "avg_loc": {"$avg": "$loc"},
        "max_loc": {"$max": "$loc"},
        "min_loc": {"$min": "$loc"},
    }}
]
stats = list(collection.aggregate(stats_pipeline))

if stats:
    s = stats[0]
    print(f"Thống kê collection '{COLLECTION}'")
    print(f"{'='*40}")
    print(f"   Tổng files:        {s['total_files']}")
    print(f"   Tổng LOC:          {s['total_loc']}")
    print(f"   Tổng nodes:        {s['total_nodes']}")
    print(f"   Tổng edges:        {s['total_edges']}")
    print(f"   LOC trung bình:    {s['avg_loc']:.1f}")
    print(f"   LOC lớn nhất:      {s['max_loc']}")
    print(f"   LOC nhỏ nhất:      {s['min_loc']}")
else:
    print("Collection rỗng — chưa có dữ liệu.")


Thống kê collection 'source_metadata'
   Tổng files:        1543
   Tổng LOC:          747512
   Tổng nodes:        435806
   Tổng edges:        1015577
   LOC trung bình:    484.5
   LOC lớn nhất:      5157
   LOC nhỏ nhất:      0


### 4.5 Top 10 files có nhiều nodes nhất

In [21]:
top_files = list(
    collection.find(
        {},
        {"file_path": 1, "loc": 1, "num_nodes": 1, "num_edges": 1, "_id": 0}
    ).sort("num_nodes", -1).limit(10)
)

if top_files:
    print(f"{'File Path':<60} {'LOC':>5} {'Nodes':>6} {'Edges':>6}")
    print(f"{'-'*60} {'-'*5} {'-'*6} {'-'*6}")
    for f in top_files:
        path = f['file_path']
        display_path = ('...' + path[-57:]) if len(path) > 60 else path
        print(f"{display_path:<60} {f['loc']:>5} {f['num_nodes']:>6} {f['num_edges']:>6}")
else:
    print("Không có dữ liệu.")


File Path                                                      LOC  Nodes  Edges
------------------------------------------------------------ ----- ------ ------
src\transformers\modeling_utils.py                            5157   2953   6548
src\transformers\trainer.py                                   4441   2753   6096
src\transformers\testing_utils.py                             4470   2718   5080
...ansformers\models\edgetam_video\modeling_edgetam_video.py  3146   2053   5144
src\transformers\models\oneformer\modeling_oneformer.py       3203   1920   4754
src\transformers\integrations\integration_utils.py            2676   1907   3765
src\transformers\models\gemma4\convert_gemma4_weights.py      2678   1901   3892
src\transformers\generation\utils.py                          3962   1896   4942
src\transformers\models\gemma4\modeling_gemma4.py             2645   1825   4480
src\transformers\tokenization_utils_base.py                   3602   1708   3953


In [22]:
client.close()
print("Đã đóng kết nối MongoDB.")


Đã đóng kết nối MongoDB.


## 5. Reflection

- **What worked**: 
  - MongoDB Spark Connector 10.3.0 tích hợp Structured Streaming ổn định.
  - `operationType="replace"` + `idFieldList="file_path"` giải quyết triệt để bài toán upsert.
  - `checkpointLocation` persistent giúp Spark khôi phục chính xác offset khi container restart.

- **Challenges**: 
  - Ban đầu dùng mode `append` mặc định → mỗi lần replay tạo document mới (ObjectID tự sinh) → trùng lặp.
  - Event thiếu trường hoặc timestamp sai format gây NullPointerException.

- **Resolution**: 
  - Chuyển sang `foreachBatch` + `replace/upsert` theo `file_path`.
  - Bổ sung bộ lọc `.where(...)` nghiêm ngặt: regex SHA-256, kiểm tra timestamp, giá trị không âm.